In [69]:
using LowLevelFEM

In [70]:
openGeometry("node-to-segment.geo")
#openPreProcessor()

In [71]:
mat_seg = Material("segment")
mat_nod = Material("node")

U = Field([mat_seg, mat_nod], type=:VectorField, dim=2, fieldName=:u)

Problem("node-to-segment", :VectorField, 2, 2, Material[Material("segment", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("node", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 6, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)

In [72]:
bc1 = BoundaryCondition("left", ux=0, uy=0)
bc2 = BoundaryCondition("right", ux=0, uy=0)
bc3 = BoundaryCondition("node", ux=0, uy=0)

BoundaryCondition("node", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [73]:
r = nodePositionVector(U)
r = projectTo2D(r)
DoFs(r)

12×1 Matrix{Float64}:
 0.0
 0.0
 1.0
 1.0
 0.0
 1.0
 0.0001414213562373095
 1.0001414213562374
 0.5
 0.5
 7.071067811865475e-5
 1.0000707106781186

In [74]:
u = applyBoundaryConditions(U, [bc1, bc2, bc3])
DoFs(u)

12×1 Matrix{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [75]:
DoFs(r+u)

12×1 Matrix{Float64}:
 0.0
 0.0
 1.0
 1.0
 0.0
 1.0
 0.0001414213562373095
 1.0001414213562374
 0.5
 0.5
 7.071067811865475e-5
 1.0000707106781186

In [76]:
C = contact(U, master="segment", slave="node")

Contact("node" -> "segment", 3 candidate nodes, 0 active, G=(6, 12), Pa=(0, 6))

In [77]:
C.G[:,:]

6×12 SparseArrays.SparseMatrixCSC{Float64, Int64} with 48 stored entries:
 -2.9672e-17  2.9672e-17  -2.9672e-17   …  -0.707107    ⋅         ⋅ 
  2.9672e-17  2.9672e-17   2.9672e-17      -0.707107    ⋅         ⋅ 
 -9.99717e-5  9.99717e-5   0.000100028     -0.707107    ⋅         ⋅ 
  9.99717e-5  9.99717e-5  -0.000100028     -0.707107    ⋅         ⋅ 
 -4.99929e-5  4.99929e-5   5.00071e-5      -0.707107  -0.707107  0.707107
  4.99929e-5  4.99929e-5  -5.00071e-5   …  -0.707107   0.707107  0.707107

In [78]:
C.G.A * r.a

6×1 Matrix{Float64}:
  0.7071067811865475
  1.1102230246251565e-16
  0.7071067811865476
  1.1102230246251565e-16
  0.7071067811865475
 -1.4139639589548597e-14

In [79]:
bc4 = BoundaryCondition("node", ux=1.1, uy=-0.9)

u = applyBoundaryConditions(U, [bc1, bc2, bc4])
updateContact!(C, u)

Contact("node" -> "segment", 3 candidate nodes, 3 active, G=(6, 12), Pa=(6, 6))

In [80]:
C.G[:,:]

6×12 SparseArrays.SparseMatrixCSC{Float64, Int64} with 48 stored entries:
 -0.0565685  0.0565685   0.0848528  …  -0.678823    ⋅         ⋅ 
  0.0565685  0.0565685  -0.0848528     -0.678823    ⋅         ⋅ 
 -0.0566285  0.0566285   0.0849928     -0.678742    ⋅         ⋅ 
  0.0566285  0.0566285  -0.0849928     -0.678742    ⋅         ⋅ 
 -0.0565985  0.0565985   0.0849228     -0.678782  -0.707107  0.707107
  0.0565985  0.0565985  -0.0849228  …  -0.678782   0.707107  0.707107

In [81]:
C.G.A * (r.a + u.a)

6×1 Matrix{Float64}:
 -0.7071067811865476
  2.7755575615628914e-15
 -0.7071067811865475
  2.886579864025407e-15
 -0.7071067811865475
  2.8088673021369575e-15

In [82]:
Cn = ∫(U ⋅ U, Γ="node")
Cn[:,:]

12×12 SparseArrays.SparseMatrixCSC{Float64, Int64} with 18 stored entries:
  ⋅    ⋅    ⋅    ⋅     ⋅          …   ⋅    ⋅    ⋅            ⋅ 
  ⋅    ⋅    ⋅    ⋅     ⋅              ⋅    ⋅    ⋅            ⋅ 
  ⋅    ⋅    ⋅    ⋅     ⋅              ⋅    ⋅    ⋅            ⋅ 
  ⋅    ⋅    ⋅    ⋅     ⋅              ⋅    ⋅    ⋅            ⋅ 
  ⋅    ⋅    ⋅    ⋅    2.66667e-5      ⋅    ⋅   1.33333e-5    ⋅ 
  ⋅    ⋅    ⋅    ⋅     ⋅          …   ⋅    ⋅    ⋅           1.33333e-5
  ⋅    ⋅    ⋅    ⋅   -6.66667e-6      ⋅    ⋅   1.33333e-5    ⋅ 
  ⋅    ⋅    ⋅    ⋅     ⋅              ⋅    ⋅    ⋅           1.33333e-5
  ⋅    ⋅    ⋅    ⋅     ⋅              ⋅    ⋅    ⋅            ⋅ 
  ⋅    ⋅    ⋅    ⋅     ⋅              ⋅    ⋅    ⋅            ⋅ 
  ⋅    ⋅    ⋅    ⋅    1.33333e-5  …   ⋅    ⋅   0.000106667   ⋅ 
  ⋅    ⋅    ⋅    ⋅     ⋅              ⋅    ⋅    ⋅           0.000106667

In [83]:
cn = 1
ct = 1

Dc = [cn 0
      0  ct]

C0 = ∫(U ⋅ Dc ⋅ U; Γ="node")

CC = subSystemMatrix(
    C0;
    onPhysicalGroup="node"
)

CC[:,:]

6×6 SparseArrays.SparseMatrixCSC{Float64, Int64} with 18 stored entries:
  2.66667e-5    ⋅          -6.66667e-6    ⋅          1.33333e-5    ⋅ 
   ⋅           2.66667e-5    ⋅          -6.66667e-6   ⋅           1.33333e-5
 -6.66667e-6    ⋅           2.66667e-5    ⋅          1.33333e-5    ⋅ 
   ⋅          -6.66667e-6    ⋅           2.66667e-5   ⋅           1.33333e-5
  1.33333e-5    ⋅           1.33333e-5    ⋅          0.000106667   ⋅ 
   ⋅           1.33333e-5    ⋅           1.33333e-5   ⋅           0.000106667

In [84]:
Ga = C.Pa * C.G
Ca = C.Pa * CC * C.Pa'

Kc = Ga' * Ca * Ga

Kc[:,:]

12×12 SparseArrays.SparseMatrixCSC{Float64, Int64} with 144 stored entries:
  1.28136e-6    2.64698e-22  -1.9226e-6    …   1.06723e-5    1.69407e-21
  2.64698e-22   1.28136e-6   -3.17637e-22      1.69407e-21   1.06723e-5
 -1.9226e-6    -3.17637e-22   2.88476e-6      -1.60132e-5   -3.38813e-21
 -3.17637e-22  -1.9226e-6     4.23516e-22     -3.38813e-21  -1.60132e-5
  2.66667e-6    3.17637e-22  -4.0e-6           1.33333e-5    2.5411e-21
  3.17637e-22   2.66667e-6   -7.41154e-22  …   2.5411e-21    1.33333e-5
  2.66949e-6    6.35275e-22  -4.0066e-6        1.33333e-5    1.69407e-21
  6.35275e-22   2.66949e-6   -6.35275e-22      1.69407e-21   1.33333e-5
 -1.53672e-5   -3.38813e-21   2.30576e-5      -0.000127992  -1.35525e-20
 -3.38813e-21  -1.53672e-5    3.38813e-21     -1.35525e-20  -0.000127992
  1.06723e-5    1.69407e-21  -1.60132e-5   …   0.000106667   2.03288e-20
  1.69407e-21   1.06723e-5   -3.38813e-21      2.03288e-20   0.000106667

In [85]:
C.Pa[:,:]

6×6 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6 stored entries:
 1.0   ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅   1.0   ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅   1.0   ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅   1.0   ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅   1.0   ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅   1.0

In [86]:
@show C.E

C.E = nothing


In [87]:
@showfields C.d

ContactVector:
  a = [-0.7071067811865476, 2.886579864025407e-15, -0.7071067811865476, 2.886579864025407e-15, -0.7071067811865475, 2.7200464103316335e-15]
  contact = Contact("node" -> "segment", 3 candidate nodes, 3 active, G=(6, 12), Pa=(6, 6))


In [88]:
C.G.A * (r.a + u.a)

6×1 Matrix{Float64}:
 -0.7071067811865476
  2.7755575615628914e-15
 -0.7071067811865475
  2.886579864025407e-15
 -0.7071067811865475
  2.8088673021369575e-15

In [89]:
bc4 = BoundaryCondition("node", ux=1.2, uy=-0.8)

u2 = applyBoundaryConditions(U, [bc1, bc2, bc4])
updateContact!(C, u)

Contact("node" -> "segment", 3 candidate nodes, 3 active, G=(6, 12), Pa=(6, 6))

In [90]:
updateContact!(C, u2)

Δd = C.G * (u2 - u)

Δd.a

6-element Vector{Float64}:
 7.589247494473767e-17
 0.14142135623730942
 7.589247494473767e-17
 0.14142135623730942
 7.589247494473767e-17
 0.14142135623730942